# Concrete and wine forest optimization

*This notebook requires a full Gurobi license and internet access for the
first OpenML download; later runs use the local cache. For a version that fits
a size-limited license, see
[Concrete Strength Optimization with Tree Ensembles](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/auto_examples/example5_concrete_strength.html).*

We first maximize a random forest's prediction of concrete compressive
strength: load the data, train a forest, formulate the optimization model,
solve it with the `"misic"` formulation, and inspect the resulting features.
We then repeat the workflow for red-wine quality. Only the final sections
compare formulations on these forests.

The applications revisit Mišić, *Optimization of Tree Ensembles*, Operations
Research 68(5):1605–1624, 2020
([doi:10.1287/opre.2019.1928](https://doi.org/10.1287/opre.2019.1928)).
This is an adaptation, not an exact reproduction: we use scikit-learn rather
than R's `randomForest`. These are surrogate-optimization examples, not
validated design recommendations.

This notebook is self-contained.

In [ ]:
import time
import warnings

import gurobipy as gp
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor

from gurobi_ml import add_predictor_constr

pd.set_option("display.float_format", "{:.3f}".format)

## Load concrete strength data and train the predictor

The UCI concrete dataset (Yeh, 1998) records ingredient quantities and curing
age, with compressive strength in MPa as the target. OpenML downloads and
caches it; download errors are not suppressed.

We use 20 trees of depth at most 5 to keep the example manageable. As in the
paper, each split considers a third of the features and each leaf contains
at least five training samples. The paper uses unlimited depth; our smaller
forests illustrate the workflow rather than reproduce its large experiments.

In [ ]:
def train_forest(X, y):
    return RandomForestRegressor(
        n_estimators=20,
        max_depth=5,
        max_features=1 / 3,
        min_samples_leaf=5,
        random_state=0,
    ).fit(X.to_numpy(), y.to_numpy())


frame = fetch_openml(data_id=4353, as_frame=True).frame
X_concrete = frame.iloc[:, :-1].astype(float)
y_concrete = frame.iloc[:, -1].astype(float)
concrete_forest = train_forest(X_concrete, y_concrete)
X_concrete.head()

## Model prediction maximization with one formulation

Let $f(x)$ be the trained forest's prediction. We choose the input vector $x$:

$$
\max_x f(x) \qquad
\text{subject to }\min_i X_{ij} \le x_j \le \max_i X_{ij}
\quad\text{for each feature }j.
$$

The bounds are the observed feature ranges. `add_predictor_constr` creates the
forest's prediction variable and constraints, and we use that variable as the
objective.

We use `"misic"`, the formulation from the paper, whose split decisions are
shared across trees, with `epsilon=2e-5` instead of the default 0. The
solution section below explains this choice.

In [ ]:
EPSILON = 2e-5


def solve(predictor, X, formulation="misic", add_constraints=None):
    """Build and solve the model; return a dict of results and the solution."""
    start = time.perf_counter()
    model = gp.Model()
    model.Params.OutputFlag = 0
    model.Params.Seed = 0
    x = model.addMVar(
        (1, X.shape[1]),
        lb=X.min().to_numpy(),
        ub=X.max().to_numpy(),
        name="features",
    )
    # "misic" and "misic_lazy" warn that a nonzero epsilon forbids an interval
    # above every threshold of the ensemble. We chose the value deliberately and
    # silence that warning.
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="epsilon=", category=UserWarning)
        prediction = add_predictor_constr(
            model, predictor, x, formulation=formulation, epsilon=EPSILON
        )
    if add_constraints is not None:
        add_constraints(model, prediction)
    model.setObjective(prediction.output.sum(), gp.GRB.MAXIMIZE)
    build_time = time.perf_counter() - start
    model.optimize()

    # Without an incumbent, only the status and the bound are available.
    solved = model.SolCount > 0
    nan = float("nan")
    result = {
        "formulation": formulation,
        "variables": model.NumVars,
        "binaries": model.NumBinVars,
        "linear constraints": model.NumConstrs,
        "general constraints": model.NumGenConstrs,
        "build [s]": build_time,
        "solve [s]": model.Runtime,
        "status": model.Status,
        "objective": model.ObjVal if solved else nan,
        "bound": model.ObjBound,
        "gap": model.MIPGap if solved else nan,
        "prediction at x": predictor.predict(x.X)[0] if solved else nan,
    }
    solution = pd.Series(x.X[0], index=X.columns, name="features") if solved else None
    return result, solution

## Solve the concrete model

`solve` builds and solves the model with a fixed solver seed (its optional
`add_constraints(model, prediction)` hook is used only in the last section).
It returns plain result data: if the solve stops early, `status`, `bound`
and `gap` show whether the incumbent is proved optimal. A solve without an
incumbent has no objective and no solution.

In [ ]:
concrete_result, concrete_solution = solve(concrete_forest, X_concrete)
pd.Series(concrete_result)

## Inspect the concrete solution

The table gives the ingredient quantities and curing age chosen by the model.
The objective is the predicted strength in MPa.

In [ ]:
concrete_solution

The objective is not a measurement, and independent feature bounds do not
describe a feasible recipe or production process.

**Why `epsilon=2e-5`.** With the default `epsilon=0`, an input exactly on a
split threshold satisfies both branches, and the solver is free to place
features on thresholds and take the better branch. The objective can then
exceed what the forest predicts at the returned input. A positive `epsilon`
removes this ambiguity: a feature that takes the right branch must be at
least `epsilon` above the threshold. Here `prediction at x` equals the
`objective` (compare them above).

The value must be larger than Gurobi's `IntFeasTol` (1e-5 by default), and
it must be large enough that `predict`, which rounds inputs to single
precision, does not round the input back onto the threshold. This choice is
fragile: single precision cannot resolve steps of `2e-5` for features of a
few hundred or more (the spacing is about 6e-5 near 700), so for some
features the margin only survives the rounding because of where the
thresholds fall. A larger value such as `1e-4` is safer, at the price of
excluding a wider interval above every threshold, which can cut off the
optimum or make the model infeasible. Even with a positive `epsilon`,
`prediction at x` is not an exactness check (the wine rows of the formulation
comparison below show why); the last section checks a solution with the
leaves selected in `tree_leaves` instead.

See [Split Thresholds and Exactness](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html#split-thresholds-and-exactness) in the user guide, and
[Tree Ensemble Formulations](Tree%20Ensemble%20Formulations.ipynb) for the
effect of `epsilon`.

## Apply the same model to red-wine quality

The UCI red-wine dataset (Cortez et al., 2009) contains physicochemical
measurements and a quality score. We train another 20-tree forest with the
same settings and solve the same bounded prediction-maximization model with
`"misic"`. The objective is now a predicted quality score rather than
strength in MPa, and each measurement is bounded by its observed range.

In [ ]:
# Quality labels are numeric, even though OpenML marks them categorical.
data = fetch_openml(data_id=40691, as_frame=True, read_csv_kwargs={"dtype": float})
X_wine = data.data.astype(float)
y_wine = data.target.astype(float)
wine_forest = train_forest(X_wine, y_wine)

## Solve and inspect the wine model

We reuse `solve`. The first table gives the solve status and objective, the
second the chosen measurements.

In [ ]:
wine_result, wine_solution = solve(wine_forest, X_wine)
display(pd.Series(wine_result))
display(wine_solution)

As with concrete, the result is a prediction at an input within the bounds.
Because the bounds are the observed ranges of each measurement separately,
the optimizer may combine extreme values that never occur together in real
wines. Compare the chosen measurements with the dataset's ranges before
reading anything into them.

## Compare formulations

The `formulation` argument of `add_predictor_constr` selects among five tree
ensemble formulations: `"leaf"` (the default, which models each tree on its
own), `"misic"` (Mišić's formulation, with split variables shared across
trees), `"misic_lazy"` (`"misic"` with lazy split constraints), `"ocean"`
(Parmentier and Vidal's) and `"biggs_perakis"` (Biggs and Perakis's
projected formulation). They represent the same function and differ in size
and strength; see [Decision Trees and Tree Ensembles](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html#decision-trees-and-tree-ensembles) in the user
guide.

We reuse the two trained forests. Only the formulation changes; the data,
objective, feature bounds and solver seed stay fixed. The table reports the
size of each model, the time to build it and to solve it, and the solution
status, objective, bound and gap.

**Runtime:** these small forests typically solve in a few seconds, but the
solves have no time limit. Larger forests or deeper trees can take
substantially longer, and a robust timing study also needs multiple seeds.

In [ ]:
FORMULATIONS = ["leaf", "misic", "misic_lazy", "ocean", "biggs_perakis"]

rows = []
for dataset, forest, X in [
    ("concrete", concrete_forest, X_concrete),
    ("winequality-red", wine_forest, X_wine),
]:
    for formulation in FORMULATIONS:
        result, _ = solve(forest, X, formulation)
        rows.append({"dataset": dataset, **result})
pd.DataFrame(rows)

Inspect `status`, `objective`, `bound`, and `gap` before comparing solve times,
and compare `prediction at x` with `objective`.
A time-limited run is not a proved optimum; a missing objective means no
incumbent was found. With `epsilon=0`, `"leaf"`, `"ocean"` and
`"biggs_perakis"` can select incompatible leaves across trees and report an
objective that no input attains. With `epsilon=2e-5`, all five formulations
reach the same objective on each forest.

For wine, `prediction at x` is lower than the objective with `"leaf"`,
`"ocean"` and `"biggs_perakis"`. These solutions put chlorides (and, for
`"ocean"`, density) exactly on a threshold, where the left branch
`x <= threshold` holds, but `predict` rounds the input to single precision,
which moves it just above the threshold and down the right branch. `epsilon`
only applies to right branches and cannot prevent this. The objective is
still attained: the leaves selected in `tree_leaves` contain a common input,
as the last section shows.

Rankings depend on the ensemble, surrounding constraints, hardware, Gurobi
version, threads, and seed. These results are not a substitute for testing
your actual decision problem.

## Advanced: check a solution with the selected leaves

`prediction at x` is not an exact check of a solution: $x$ can sit exactly on
a split threshold, and `predict` rounds its input to single precision, which
can send $x$ down the other branch. Instead, we intersect the boxes of inputs
that reach the leaves selected in `prediction.tree_leaves`, one per tree. If
the intersection is empty, no input attains the objective. Otherwise, we move
$x$ to the closest single-precision point inside it, like rounding a MIP
solution, and evaluate `predict` there. If $x$ is subject to other
constraints, check them at the moved point too.

In [ ]:
def leaf_box(tree, leaf):
    """Bounds lo < x <= hi of the inputs reaching a leaf of a scikit-learn tree."""
    lo = np.full(tree.n_features, -np.inf)
    hi = np.full(tree.n_features, np.inf)
    parent = {}
    for node in range(tree.node_count):
        if tree.children_left[node] >= 0:
            parent[tree.children_left[node]] = (node, True)
            parent[tree.children_right[node]] = (node, False)
    node = leaf
    while node in parent:
        node, went_left = parent[node]
        feature, threshold = tree.feature[node], tree.threshold[node]
        if went_left:
            hi[feature] = min(hi[feature], threshold)
        else:
            lo[feature] = max(lo[feature], threshold)
    return lo, hi


def selected_box(forest, prediction):
    """Intersection of the boxes of the leaves selected in prediction."""
    lo = np.full(forest.n_features_in_, -np.inf)
    hi = np.full(forest.n_features_in_, np.inf)
    for estimator, leaves in zip(forest.estimators_, prediction.tree_leaves):
        # One column of leaves.variables per reachable leaf, whose node id in
        # the scikit-learn tree is in leaves.nodes; row 0 is our only input.
        selected = leaves.nodes[np.argmax(leaves.variables.X[0])]
        box_lo, box_hi = leaf_box(estimator.tree_, selected)
        lo = np.maximum(lo, box_lo)
        hi = np.minimum(hi, box_hi)
    return lo, hi


def project_to_box(x, lo, hi):
    """The closest single-precision point with lo < x <= hi, or None."""
    lo32 = lo.astype(np.float32)
    lo32 = np.where(lo32 > lo, lo32, np.nextafter(lo32, np.float32(np.inf)))
    hi32 = hi.astype(np.float32)
    hi32 = np.where(hi32 <= hi, hi32, np.nextafter(hi32, np.float32(-np.inf)))
    if np.any(lo32 > hi32):
        return None
    return np.clip(x.astype(np.float32), lo32, hi32)


def check_leaves(forest, prediction, x):
    """Check the solution x with the leaves selected in prediction.

    Return the results and x moved into the selected leaves, or None if no
    input reaches them.
    """
    moved = project_to_box(x, *selected_box(forest, prediction))
    if moved is None:
        results = {
            "leaves reachable": False,
            "max shift": np.nan,
            "prediction at moved x": np.nan,
        }
        return results, None
    results = {
        "leaves reachable": True,
        "max shift": np.abs(moved - x).max(),
        "prediction at moved x": forest.predict([moved])[0],
    }
    return results, moved

In [ ]:
def solve_and_check(forest, X, formulation):
    """Solve with a formulation and check the selected leaves."""
    # solve does not return the prediction object: keep it with the hook.
    kept = {}

    def keep_prediction(model, prediction):
        kept["prediction"] = prediction

    result, solution = solve(forest, X, formulation, add_constraints=keep_prediction)
    row = {
        "objective": result["objective"],
        "prediction at x": result["prediction at x"],
    }
    if solution is None:  # no incumbent to check
        return row
    check, _ = check_leaves(forest, kept["prediction"], solution.to_numpy())
    return {**row, **check}


checks = []
for dataset, forest, X in [
    ("concrete", concrete_forest, X_concrete),
    ("winequality-red", wine_forest, X_wine),
]:
    for formulation in FORMULATIONS:
        checks.append(
            {
                "dataset": dataset,
                "formulation": formulation,
                **solve_and_check(forest, X, formulation),
            }
        )
# A general float format shows the small shifts.
with pd.option_context("display.float_format", "{:.4g}".format):
    display(pd.DataFrame(checks))

The selected leaves are reachable for every formulation on both forests, and
the prediction at the moved $x$ equals the objective (68.69 for concrete,
7.094 for wine), also in the wine rows where the prediction at $x$ is lower.
The largest shift, 3.05e-5 for concrete, is half the single-precision spacing
of the aggregate quantities; for wine, the shifts are below 5e-7.

For an application that adds constraints on selected tree leaves, see
[Drug Design with Tree Ensembles](../tree-ensembles-merck/Drug%20Design%20with%20Tree%20Ensembles.ipynb).

Copyright © 2023-2026 Gurobi Optimization, LLC